In [5]:
from srai.loaders import HFLoader
import json
import os
import geopandas as gpd
from shapely import LineString, MultiPoint
import pandas as pd
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [6]:
token = os.environ["HF_access_token"]

In [7]:
hf_loader = HFLoader(hf_token=token)

In [8]:
data = hf_loader.load(dataset_name="nyc_bike", name="nyc_bike_2014")

In [11]:
def make_geodataframe_previous(dataframe: pd.DataFrame) -> gpd.GeoDataFrame:
    start_station_geometry = gpd.points_from_xy(
        x=dataframe["start station longitude"], y=dataframe["start station latitude"]
    )
    end_station_geometry = gpd.points_from_xy(
        x=dataframe["end station longitude"], y=dataframe["end station latitude"]
    )
    multi_point_stations_geometries = [
        MultiPoint([start, end])
        for start, end in zip(start_station_geometry, end_station_geometry)
    ]
    gdf = gpd.GeoDataFrame(
        dataframe.drop(
            [
                "start station latitude",
                "start station longitude",
                "end station latitude",
                "end station longitude",
            ],
            axis=1,
        ),
        geometry=multi_point_stations_geometries,
        crs="EPSG:4326",
    )

    return gdf

In [12]:
gdf = make_geodataframe_previous(data)

In [7]:
data.head()

,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,391,2013-09-30 07:41:55,2013-09-30 07:48:26,438,St Marks Pl & 1 Ave,40.727791,-73.985649,497,E 17 St & Broadway,40.737050,-73.990093,20255,Subscriber,1979,1
1,570,2013-09-30 07:41:56,2013-09-30 07:51:26,453,W 22 St & 8 Ave,40.744751,-73.999154,368,Carmine St & 6 Ave,40.730386,-74.002150,19000,Subscriber,1955,2
2,1043,2013-09-30 07:41:56,2013-09-30 07:59:19,388,W 26 St & 10 Ave,40.749718,-74.002950,456,E 53 St & Madison Ave,40.759711,-73.974023,15311,Subscriber,1972,1
3,304,2013-09-30 07:41:59,2013-09-30 07:47:03,336,Sullivan St & Washington Sq,40.730477,-73.999061,382,University Pl & E 14 St,40.734927,-73.992005,19305,Subscriber,1964,1
4,368,2013-09-30 07:42:00,2013-09-30 07:48:08,248,Laight St & Hudson St,40.721854,-74.007718,327,Vesey Pl & River Terrace,40.715338,-74.016584,17882,Subscriber,1971,1


In [8]:
data = hf_loader.load(dataset_name="geolife")

c:\Users\Kacper Kozaczko\Desktop\Stuff\PWr\III_semestr\srai\venv\lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for kraina/geolife contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/kraina/geolife
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [13]:
data["geometry"] = data["arrays_geometry"].map(LineString)
geolife = gpd.GeoDataFrame(
    data.drop(["arrays_geometry"], axis=1),
    geometry=data["geometry"],
    crs="EPSG:4326",
)

C:\Users\Kacper Kozaczko\AppData\Local\Temp\ipykernel_7288\1011287707.py:1: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  data["geometry"] = data["arrays_geometry"].map(LineString)


In [14]:
geolife.head()

,latitude,longitude,altitude,time,mode,trajectory_id,user_id,geometry
0,"[39.988992, 39.990964, 39.993207]","[116.327023, 116.327041, 116.326827]","[128.937004593176, 221.128615485564, 217.19159...","[2000-01-01 23:12:19, 2000-01-01 23:13:21, 200...","[unknown, unknown, unknown]",20000101231219,163,"LINESTRING (116.32702 39.98899, 116.32704 39.9..."
1,"[39.9742333333333, 39.9743166666667, 39.974466...","[116.330383333333, 116.33045, 116.33045, 116.3...","[823.490813648294, 823.490813648294, 741.46981...","[2007-04-12 09:31:32, 2007-04-12 09:39:37, 200...","[unknown, unknown, unknown, unknown, unknown, ...",20070412093132,142,"LINESTRING (116.33038 39.97423, 116.33045 39.9..."
2,"[39.9755166666667, 39.97585, 39.9759833333333,...","[116.330283333333, 116.3304, 116.330466666667,...","[351.049868766404, 114.829396325459, 114.82939...","[2007-04-12 10:18:53, 2007-04-12 10:20:15, 200...","[unknown, unknown, bike, bike, walk]",20070412101853,161,"LINESTRING (116.33028 39.97552, 116.33040 39.9..."
3,"[39.9764666666667, 39.9764, 39.97625, 39.9762,...","[116.330066666667, 116.33015, 116.330266666667...","[173.884514435696, 173.884514435696, 173.88451...","[2007-04-12 10:21:16, 2007-04-12 10:21:22, 200...","[bike, bike, bike, bike, bike, bike, bike, bik...",20070412102116,163,"LINESTRING (116.33007 39.97647, 116.33015 39.9..."
4,"[39.97585, 39.9759833333333, 39.9761, 39.97623...","[116.3304, 116.330466666667, 116.3305, 116.330...","[114.829396325459, 114.829396325459, 118.11023...","[2007-04-12 10:23:25, 2007-04-12 10:24:37, 200...","[walk, walk, walk, walk]",20070412102325,161,"LINESTRING (116.33040 39.97585, 116.33047 39.9..."
